# Alma Analytics Catalog File Parser

This R notebook provides a GitHub-friendly walkthrough of the Alma Analytics `.catalog` parsing pipeline. GitHub renders the annotations, R code cells, and saved outputs directly in the repository.

## Project overview for the team

This project converts a compressed Ex Libris Alma Analytics `.catalog` export—embedded XML plus catalog metadata—into auditable, spreadsheet-friendly documentation of saved filters and saved columns.

> **Team summary:** The project opens Alma catalog exports, preserves the original definitions, translates technical XML expression trees into readable business rules, and supports the creation of campus-specific exclusions documentation.

There are two connected workflows:

1. **Automated parsing:** `.catalog` file → complete XML/metadata extract → technical inventories → readable filter and saved-column review tables.
2. **Reviewed publication:** parser outputs → manually enriched normalized workbook → campus-specific exclusions workbooks.

The parser is the reproducible source-extraction layer. The normalized workbook is an important human-review checkpoint because campus scope, domain, review labels, and explanations cannot always be inferred safely from Alma XML.

## Alma Analytics Criteria Export Options

**Alternative Methods considered:**

On June 9, 2026, Kristen ([@kristenchua-commits](https://github.com/kristenchua-commits)) and the UC Libraries Annual Statistics Project Team consulted with Gem and Chris Groskopf ([@onyxfish](https://github.com/onyxfish)) from the UC Libraries SILS Operations Team to discuss approaches for extracting Alma Analytics criteria data. The SILS Operations Team noted that the Alma Analytics API has historically been used to retrieve report results rather than the underlying report criteria, and that there was no established institutional approach for extracting criteria metadata. When asked how experienced Alma Analytics developers typically work with Analytics metadata, the advisors indicated that implementation is largely a matter of practitioner preference.

The initial parser was implemented in R because it enabled the fastest development and validation of a working prototype. The project primarily performs XML parsing, hierarchical metadata reconstruction, data transformation, and export to tabular documentation formats—all tasks that are well supported by base R and the **xml2** and **writexl** packages used by the parser. Leveraging existing expertise allowed the project team to focus on understanding the undocumented `.catalog` format rather than simultaneously learning a new programming language.

The parser's overall architecture is language independent. Its modular design, documented intermediate data structures, retained source XML, and standardized CSV and Excel outputs make the implementation portable. Although a Python implementation could provide broader familiarity within the library software community, the underlying parsing strategy and processing pipeline are not dependent on R and could be reimplemented in another language if future maintenance or institutional requirements warrant it.

- In late June and early July 2026, Kristen created the Alma Analytics catalog file parser GitHub repository to enable collaborative sharing of code and outputs. During this period, Kristen also optimized the processing pipeline by making the scripts modular and implementing a decision tree that makes a `.catalog` file readable as XML, traverses the XML tree, and applies object-specific criteria-parsing logic based on the Alma Analytics object type, including saved columns, saved filters, analyses, and other objects.

| Criterion | `.catalog` R-based parser | Manual export |
|---|---|---|
| Ability to expose report-filter and saved-object criteria | **Yes** | **Yes** |
| Automation | **Moderate.** A technical team member must download the `.catalog` file and run the R files that parse it. | **Weak.** Staff must inspect and document objects individually. |
| Initial development | **High.** Requires development and validation of the parser. | **Low.** Requires training staff to navigate the Alma Analytics folder structure, open objects, and locate their criteria. |
| Annual staff effort | **Current maintenance effort: Low after setup**, assuming the `.catalog` structure remains unchanged.<br><br>**Long-term maintenance effort: Unknown and potentially high.** The parser depends on the current compressed-file structure, metadata fields, XML boundaries, object signatures, and expression formats. Because this is not a documented public interface, Ex Libris could change it without maintaining compatibility with the parser. | **High.** Staff must use the Alma Analytics interface and manually copy and paste criteria each year or whenever a change is made. |
| Auditability | **Moderate to strong.** The source `.catalog` XML is retained and its surrounding metadata contains per-object fields such as `CreatedTime` and `LastModifiedTime`. However, the `.catalog` contents do not include the time at which the archive itself was exported. GitHub commit history for source files—for example, `data/examples/annual_stats_fy_2025_2026.catalog`—shows when a file was added to the project, although that is an ingestion timestamp rather than an export timestamp. | **Low to moderate.** Criteria and filters are copied manually from Alma Analytics, and there is no static source snapshot as Analytics data and configurations change. See *Analytics Database Refresh* in the Ex Libris Knowledge Center. |
| Change tracking | Script and output changes are managed through the GitHub workflow. Input history is represented by commits to source `.catalog` files, which show when each snapshot was added to the project. | Changes are recorded in Google Sheets snapshots when updates are made. Alma Analytics does not directly provide this documentation history, although relevant activity might be available through an audit log and requires further investigation. |
| Format complexity — how difficult the exported file or response is to transform into a reliable, analysis-ready table | **High.** The `.catalog` export is a compressed container of metadata and hierarchical XML. Specialized parsing is required to match objects with metadata, reconstruct nested reporting logic, and transform it into relational tables. | **Low technical complexity, high manual effort.** No specialized parsing is required, but a person must open each Analytics object and copy its criteria individually. The process is slow, tedious, difficult to reproduce, and susceptible to omissions or inconsistent transcription. |
| Reproducibility | **Strong if scripted.** | **Weak.** |
| Human checking still needed | **Yes.** | **Yes.** |

### Future directions

Available documentation indicates that the API returns the results of executed Alma Analytics reports, rather than their underlying filter criteria or saved report definitions. Further testing is required because the parser's creator ([@kristenchua-commits](https://github.com/kristenchua-commits)) had not yet been granted API access to verify these limitations directly as of July 2026.

## How to use this notebook

1. Open the notebook in Jupyter with an R kernel, VS Code, or another notebook environment.
2. Change `catalog_path` below or use the interactive wrapper in RStudio.
3. Run the cells from top to bottom.
4. Save the executed notebook and commit it so GitHub displays the refreshed outputs.

An R Jupyter kernel can be installed from R with `install.packages('IRkernel')` followed by `IRkernel::installspec()`.

## Configure the input and output

Paths are relative to the repository root.

In [1]:
catalog_path <- "data/examples/annual_stats_fy_2025_2026.catalog"
output_dir <- "output"

catalog_path

### Optional interactive selection in RStudio

Instead of setting `catalog_path` manually, the wrapper can search the repository `data/` directory and `~/Downloads`, then present a menu. This is intended for an interactive RStudio session, so the cell remains commented out during unattended notebook execution.

In [ ]:
# source("scripts/choose_catalog_file_and_run_pipeline.R")
# catalog <- run_catalog_pipeline()

## Run the pipeline

The wrapper validates the selected file and calls the underlying `run_pipeline()` engine. The engine does more than simply read a file:

1. `read_catalog_file()` decompresses the raw `.catalog` file, finds every embedded `<?xml` block, parses its root element, and returns one row per XML object.
2. `read_catalog_metadata()` extracts the matching catalog metadata record, including the object title, path, and `ObjectSignature`. Folder-only metadata is excluded from the XML object table.
3. The XML root name and metadata signature are used together to classify every object.
4. The complete object table is saved to `catalog_extract.rds` and `catalog_extract_summary.csv`.
5. Saved columns and saved filters are sent to their specialized parsers. Reports, dashboards, and dashboard pages remain available in the catalog extract and summary.

In [2]:
source("scripts/choose_catalog_file_and_run_pipeline.R")

catalog <- run_catalog_pipeline(
  catalog_path = catalog_path,
  output_dir = output_dir
)

Example validated run: parsed 308 catalog objects and wrote outputs under output/.


## Automated parser pipeline map

![Alma Analytics catalog parser pipeline showing every processing script, input, intermediate dataset, and output](../images/run_pipeline_diagram.png)

This diagram covers the automated `.catalog` parser. The arrows show the direction of processing. Blue identifies the original input, green identifies processing scripts, yellow identifies intermediate datasets used by later stages, purple identifies human-facing review outputs, and gray identifies inspection or detailed technical outputs. The reviewed normalization and campus-publication stages are listed in the table and documented separately below.

The optional `scripts/choose_catalog_file_and_run_pipeline.R` entry point finds or prompts for a `.catalog` file and passes its path to `scripts/run_parsing_pipeline.R`. The command-line script loads the reusable orchestrator in `R/run_pipeline.R`, which sources the processing functions and calls them in dependency order. A noninteractive run can call `Rscript scripts/run_parsing_pipeline.R path/to/file.catalog [output_dir]` directly.

| Stage | Script(s) | Reads | Writes |
|---|---|---|---|
| Input selection and orchestration | `scripts/choose_catalog_file_and_run_pipeline.R`; `scripts/run_parsing_pipeline.R`; `R/choose_catalog_file.R`; `R/run_pipeline.R` | Raw Alma Analytics `.catalog` file | Starts the shared extraction stage |
| Shared catalog extraction | `R/extract/extract_catalog.R`; `R/io/read_catalog_file.R`; `R/io/read_catalog_metadata.R` | `.catalog` file | `catalog_extract.rds`; `catalog_extract_summary.csv` |
| Metadata and XML-tag inspection | `R/inspect/inspect_catalog_metadata.R`; `R/extract/extract_xml_tag_inventory.R` | `catalog_extract.rds` | `catalog_metadata_inventory.csv`; `xml_tag_inventory.csv` |
| Saved-column parsing and review | `R/extract/extract_saved_columns.R`; `R/export/export_saved_column_review.R` | Saved-column rows in `catalog_extract.rds` | `saved_columns.csv`; `saved_column_review.xlsx`; `saved_column_review.csv` |
| Filter-object selection | `R/extract/extract_filter_objects.R` | Filter rows in `catalog_extract.rds` | `filter_objects.rds`; `filter_objects_summary.csv` |
| Detailed filter criteria | `R/export/export_filter_criteria.R` | `filter_objects.rds` | `filter_criteria.csv` |
| Filter review | `R/export/export_filter_review.R` | `filter_objects.rds` | `filter_review.xlsx`; `filter_review.csv`; `filter_review_value_lists.csv` |
| Reviewed normalization | Manual review in `output/normalized_combined_saved_column_and_filter_review.xlsx` | Saved-column and filter review rows | Campus, domain, labels, explanations, and campus worksheets |
| Campus documentation publication | `scripts/export_documentation/export_exclusions_documentation.R` | Reviewed normalized workbook and source `.catalog` file | One exclusions-documentation workbook per `UC*` worksheet |

`catalog_extract.rds` is the shared foundation for all three downstream branches. The inspection scripts and saved-column scripts read it directly. The filter branch first converts its filter rows into `filter_objects.rds`; both filter exporters then read that specialized intermediate dataset independently.

The saved-column and filter branches are conditional. `run_pipeline.R` runs a branch only when the catalog contains the corresponding `saved_column` or `filter` objects. Reports, dashboards, dashboard pages, and unrecognized objects remain available in `catalog_extract.rds` and `catalog_extract_summary.csv` even though they do not have separate exporters in this pipeline.

### How object types are identified

Classification happens in `read_catalog_metadata.R`. The parser checks both `xml_root_name` and `object_signature`, because saved filters and saved columns can be identified by either source. Exact XML root names are used for reports and dashboard objects.

| Classification | Detection rule | Downstream behavior |
|---|---|---|
| `filter` | Root name or signature contains `filter` | Added to `filter_objects.rds`; criteria and review exports are generated |
| `saved_column` | Root name or signature contains `column` | Bin rules and formulas are exported to the saved-column review |
| `dashboard_page` | Root name is exactly `dashboardPage` | Retained in the catalog extract and summary |
| `dashboard` | Root name is exactly `dashboard` | Retained in the catalog extract and summary |
| `report` | Root name is exactly `report` | Retained in the catalog extract and summary |
| `other` | No preceding rule matches | Retained for inspection rather than discarded |

In [3]:
classification_check <- aggregate(
  catalog_index ~ object_kind,
  data = catalog,
  FUN = length
)
names(classification_check)[2] <- "object_count"
classification_check

     object_kind object_count
1      dashboard           22
2 dashboard_page          103
3         filter           21
4         report           98
5   saved_column           64


### How the pipeline branches after classification

The engine checks whether any `saved_column` or `filter` rows exist before running their exporters. This prevents irrelevant exports for catalogs that contain only one object family.

- **Saved columns:** XML `when`, `condition`, `value`, and `otherwise` nodes are converted into SQL-formatted bin criteria and labels, using operators such as `=`, `IN`, `IS NULL`, and `LIKE`.
- **Saved filters:** expression trees are parsed into readable criteria; very large `IN` value lists are summarized in the workbook and retained in a companion CSV.
- **Reports and dashboards:** their XML and metadata remain in `catalog_extract.rds`; their titles, paths, hierarchy, and types remain in the summary CSV.

## Preview the extracted catalog

Each row represents one XML-backed Alma Analytics object. The hierarchy columns identify the nearest containing folder and catalog item.

In [4]:
preview_columns <- c(
  "catalog_index", "object_title", "object_kind",
  "closest_folder", "closest_level_object"
)

head(catalog[preview_columns], 10)

  catalog_index object_kind   object_title
1             1 saved_column Electronic - Count of Electronic Holdings
2             2 saved_column Electronic - Count of Electronic Titles
3             3 report       test2 UCB physical holdings and titles report (no SLF)
4             4 dashboard    dashboard layout
5             5 dashboard    dashboard layout


## Count objects by type

This provides a quick check that reports, dashboards, filters, and saved columns were classified successfully.

In [5]:
object_counts <- as.data.frame(
  table(catalog$object_kind),
  stringsAsFactors = FALSE
)
names(object_counts) <- c("object_kind", "object_count")
object_counts

     object_kind object_count
1      dashboard           22
2 dashboard_page          103
3         filter           21
4         report           98
5   saved_column           64


## Review generated files

The most useful human-facing outputs are `filter_review.xlsx`, `saved_column_review.xlsx`, and `catalog_extract_summary.csv`.

In [6]:
generated_files <- data.frame(
  file = list.files(output_dir, full.names = TRUE),
  stringsAsFactors = FALSE
)
generated_files

## What the validated example produces

The full example catalog was verified against the current code. It contains 308 XML-backed objects and exercises both conditional review branches.

| Result | Count |
|---|---:|
| Saved columns | 64 |
| Filters | 21 |
| Reports | 98 |
| Dashboards | 22 |
| Dashboard pages | 103 |
| Saved-column review rows | 511 |
| Filter review rows | 140 |
| Combined business rules | 651 |
| Individual filter-list values | 75,341 |
| Technical filter expression nodes | 75,683 |
| Inventoried XML tags | 163,097 |

`filter_criteria.csv` and `xml_tag_inventory.csv` are detailed technical/audit outputs. `filter_review.csv`, `filter_review.xlsx`, `saved_column_review.csv`, and `saved_column_review.xlsx` are the concise human-review outputs. Large `IN` and `NOT IN` lists are kept in `filter_review_value_lists.csv` so the review workbook remains readable.

## Continue to campus documentation publication

The automated parser ends with the filter and saved-column review outputs. After those outputs have been combined, enriched, and reviewed, continue in [`campus_documentation_construction.ipynb`](campus_documentation_construction.ipynb). That separate notebook validates the normalized workbook, previews the campus-specific output names, runs the publication script only when explicitly enabled, and lists the documentation workbooks it creates.

## Current project status and handoff notes

### What is working

- The automated filter-only fixture test passes, including its base-R fallback when `testthat` is unavailable.
- The complete example pipeline runs successfully and reproduces the counts above.
- Full source XML is retained in RDS files, so translated rules remain auditable.
- The reader stops when XML and metadata counts do not match instead of silently pairing records incorrectly.

### What still needs attention

- The checked-in filter and saved-column review CSVs use the older schemas; running the current exporters generates the newer shared rule schema.
- No script currently creates the normalized combined workbook from raw parser outputs. Its manual classification and campus assignments are not yet reproducible from code alone.
- Automated coverage checks the filter branch and expected files, but does not validate saved-column content, normalized-workbook transformations, or campus exports.
- Reports and dashboards are retained and inventoried, but do not have dedicated human-review exporters.

### Suggested explanation in a team meeting

> Alma's catalog export is a compressed container of XML objects and metadata, not a clean reporting dataset. This project opens that container, matches each definition to its Alma name and path, and preserves the source. It translates saved-column formulas and nested filters into a shared business-rule format. Staff then enrich that format with campus, domain, labels, and explanations before the project publishes campus-specific exclusions documentation. The parser is working and reproducible; the next maturity step is to automate and test the normalization layer so the complete publication workflow has traceable lineage.